In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors


In [3]:
tracks = pd.read_csv(
    "../data/raw/fma_metadata/tracks.csv",
    index_col=0,
    header=[0, 1]
)

features = pd.read_csv(
    "../data/raw/fma_metadata/features.csv",
    index_col=0,
    header=[0, 1, 2]
)

In [4]:
small_tracks = tracks[
    tracks[("set", "subset")] == "small"
].copy()

small_features = features.loc[small_tracks.index].copy()

In [5]:
small_tracks.shape


(8000, 52)

In [6]:
small_features.shape

(8000, 518)

In [7]:
small_features.isna().sum().sum()

np.int64(0)

In [8]:
small_features.dtypes.value_counts()

float64    518
Name: count, dtype: int64

In [9]:
# scale the features

scaler = StandardScaler()

X_scaled = scaler.fit_transform(small_features)

In [10]:
X_scaled.shape

(8000, 518)

In [11]:
X_scaled.std()

np.float64(1.0)

In [12]:
def recommend_random_tracks(track_id, n=5, random_state=None):

    if track_id not in small_tracks.index:
        raise ValueError("Track ID not found in FMA Small.")

    seed_artist = small_tracks.loc[
        track_id, ("artist", "name")
    ]

    # Exclude seed and same artist
    candidates = small_tracks[
        (small_tracks.index != track_id)
        &
        (small_tracks[("artist", "name")] != seed_artist)
    ]

    sample = candidates.sample(
        n=n,
        random_state=random_state
    )

    results = sample[
        [
            ("artist", "name"),
            ("track", "title"),
            ("track", "genre_top")
        ]
    ].copy()

    results.columns = [
        "artist",
        "title",
        "genre"
    ]

    return results

In [13]:
recommend_random_tracks(
    140,
    n=5,
    random_state=42
)

,artist,title,genre
track_id,,,
115775,Jelsonic,Vapour Trails,Instrumental
59726,The Rosen Sisters,Gravel Walk,International
83437,Keaton Henson,You Don't Know How Lucky You Are,Folk
121739,Frozen Houses,Birmingham Road Birthday Song featuring Lucy Hill,Pop
143057,Mystery Mammal,There Is Something Wrong,Electronic


In [15]:
#
test_track_ids = [
    140,
    54033,
    59876,
    42234,
    68893,
    43883,
    123978,
    149701,
    63471,
    17608
]

In [16]:
random_rows = []

for i, seed_id in enumerate(test_track_ids):

    seed = small_tracks.loc[seed_id]

    recommendations = recommend_random_tracks(
        seed_id,
        n=5,
        random_state=42 + i
    )

    for rec_id, rec in recommendations.iterrows():

        random_rows.append({
            "seed_track_id": seed_id,
            "seed_artist": seed[("artist", "name")],
            "seed_title": seed[("track", "title")],
            "seed_genre": seed[("track", "genre_top")],

            "recommended_track_id": rec_id,
            "recommended_artist": rec["artist"],
            "recommended_title": rec["title"],
            "recommended_genre": rec["genre"],

            "manual_similarity_1_5": np.nan,
            "connection_type": "",
            "notes": ""
        })

In [17]:
random_eval = pd.DataFrame(random_rows)

random_eval.shape

(50, 11)

In [18]:
random_eval.to_csv(
    "../data/processed/random_manual_evaluation.csv",
    index=False
)

# Calculate random genre agreement@10

In [21]:
def random_genre_agreement_at_k(
    tracks_df,
    k=10,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    genres = tracks_df[("track", "genre_top")].to_numpy()
    artists = tracks_df[("artist", "id")].to_numpy()

    scores = []

    all_indices = np.arange(len(tracks_df))

    for i in range(len(tracks_df)):

        if pd.isna(genres[i]):
            continue

        valid_candidates = all_indices[
            (all_indices != i)
            &
            (artists != artists[i])
        ]

        sampled = rng.choice(
            valid_candidates,
            size=k,
            replace=False
        )

        sampled_genres = genres[sampled]

        valid = ~pd.isna(sampled_genres)

        if valid.sum() == 0:
            continue

        score = (
            sampled_genres[valid] == genres[i]
        ).mean()

        scores.append(score)

    return np.mean(scores)

In [20]:
random_genre_agreement = (
    random_genre_agreement_at_k(
        small_tracks,
        k=10
    )
)

random_genre_agreement

np.float64(0.12397500000000003)